# 📈 Trading Bot — Backtest de Estrategia Sistemática en Acciones Small Cap

**Autor:** Henry Paolo Alfaro Sotil  
**GitHub:** [github.com/elbrujo325](https://github.com/elbrujo325)  
**Contacto:** henry.alfaro1@unmsm.edu.pe

---

## Descripción del Proyecto

Este proyecto implementa un sistema completo de análisis cuantitativo y backtesting para una estrategia de trading sistemático en activos Small Cap del mercado estadounidense. Demuestra competencias en:

- **Extracción y preparación de datos** con `yfinance` y `pandas`
- **Feature Engineering** financiero: ATR, SMA, ROC, análisis de estructura de precios
- **Simulación de operaciones** (backtest discreto con gestión de riesgo)
- **Evaluación estadística de estrategias**: Win Rate, Profit Factor, Sharpe Ratio, Max Drawdown
- **Visualización analítica** de resultados y métricas de rendimiento

> ⚠️ **Aviso:** Este proyecto es exclusivamente un ejercicio de análisis de datos y portafolio. No constituye asesoría financiera.

---

## Lógica de la Estrategia

| Parámetro | Valor |
|-----------|-------|
| Universo | Acciones Small Cap (precio $1–$20) |
| Timeframe | 1 hora |
| Indicadores | ATR(50), SMA(10), ROC(5), Estructura de velas |
| Stop Loss | 1.9× ATR |
| Take Profit | 3.2× ATR |
| Time Exit | 40 velas máximo |
| Riesgo por operación | 1% del capital |
| Capital inicial | $10,000 |

**Condición de entrada (Long):**
- Precio entre $1.00 y $20.00
- Precio de apertura > SMA(10)
- ROC decelerando (momentum débil previo)
- Estructura de precios favorable (Open[7] > High[9])


## 1. Importación de Librerías y Configuración


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

# ── Configuración de la estrategia ──────────────────────────────
TICKET      = "ABSI"       # Activo a analizar
CAPITAL     = 10_000       # Capital inicial en USD
RIESGO_FIJO = CAPITAL * 0.01  # Riesgo por operación: 1%
C_SL        = 1.9          # Multiplicador Stop Loss  (× ATR)
C_TP        = 3.2          # Multiplicador Take Profit (× ATR)
MAX_VELAS   = 40           # Máximo de velas por operación (Time Exit)
PERIODO_ATR = 50
PERIODO_SMA = 10
PERIODO_ROC = 5

print(f"Activo: {TICKET} | Capital: ${CAPITAL:,} | Riesgo/op: ${RIESGO_FIJO:.0f} (1%)")
print(f"SL: {C_SL}×ATR | TP: {C_TP}×ATR | R:R = 1:{C_TP/C_SL:.2f}")


## 2. Descarga y Preparación de Datos


In [ ]:
# Descarga de datos históricos
data = yf.Ticker(TICKET)
df = data.history(period="max", interval="1h")

print(f"Datos descargados: {len(df):,} velas")
print(f"Período: {df.index[0].date()} → {df.index[-1].date()}")
print(f"\nPrimeras filas:")
df.head(3)


## 3. Feature Engineering — Indicadores Técnicos


In [ ]:
def calcular_atr(df, periodo):
    """Average True Range: mide la volatilidad del activo."""
    hl  = df['High'] - df['Low']
    hc  = abs(df['High'] - df['Close'].shift(1))
    lc  = abs(df['Low']  - df['Close'].shift(1))
    tr  = pd.concat([hl, hc, lc], axis=1).max(axis=1)
    return tr.rolling(window=periodo).mean()

def calcular_sma(df, periodo):
    """Simple Moving Average (tendencia)."""
    return df['Close'].rolling(window=periodo).mean()

def calcular_roc(df, periodo):
    """Rate of Change: velocidad del momentum de precios."""
    return ((df['Close'] / df['Close'].shift(periodo)) - 1) * 100

def calcular_estructura(df):
    """Condición de estructura: Open[7] > High[9] (ruptura de rango previo)."""
    return df['Open'].shift(7) > df['High'].shift(9)

def calcular_sl_tp(precio_entrada, atr, c_sl, c_tp):
    """Calcula precios de Stop Loss y Take Profit basados en ATR."""
    return precio_entrada - atr * c_sl, precio_entrada + atr * c_tp

def calcular_size(atr, riesgo_fijo, c_sl):
    """Position sizing por riesgo fijo: size = riesgo / distancia_SL."""
    distancia_sl = c_sl * atr
    return int(riesgo_fijo / distancia_sl) if distancia_sl > 0 else 0


# Aplicar indicadores al DataFrame
df["ATR"]          = calcular_atr(df, PERIODO_ATR)
df["SMA"]          = calcular_sma(df, PERIODO_SMA)
df["ROC"]          = calcular_roc(df, PERIODO_ROC)
df["Estructura_OK"] = calcular_estructura(df)

print("Indicadores calculados:")
print(df[["ATR", "SMA", "ROC", "Estructura_OK"]].tail(3))


## 4. Generación de Señales y Backtest


In [ ]:
# ── Señal de entrada ────────────────────────────────────────────
df["Signal"] = (
    (df['Close'] > 1.0) & (df['Close'] <= 20.0) &   # Rango Small Cap
    (df['Open']  > df["SMA"]) &                      # Precio sobre tendencia
    (df["ROC"]   < df["ROC"].shift(3)) &             # Momentum desacelerando
    (df["Estructura_OK"] == True)                    # Ruptura de estructura
)

print(f"Total señales generadas: {df['Signal'].sum()}")

# ── Loop de Backtest ─────────────────────────────────────────────
en_posicion  = False
trades       = []

for i in range(1, len(df)):
    if not en_posicion and df["Signal"].iloc[i - 1]:
        p_entrada  = df["Open"].iloc[i]
        atr_v      = df["ATR"].iloc[i - 1]
        size       = calcular_size(atr_v, RIESGO_FIJO, C_SL)
        precio_sl, precio_tp = calcular_sl_tp(p_entrada, atr_v, C_SL, C_TP)

        en_posicion  = True
        fecha_in     = df.index[i]
        mae_temp     = p_entrada
        mfe_temp     = p_entrada
        conteo_velas = 0

    elif en_posicion:
        conteo_velas += 1
        high_act  = df["High"].iloc[i]
        low_act   = df["Low"].iloc[i]
        close_act = df["Close"].iloc[i]

        mae_temp = min(mae_temp, low_act)
        mfe_temp = max(mfe_temp, high_act)

        salida = False
        if low_act <= precio_sl:
            p_salida, motivo = precio_sl, "SL"
            salida = True
        elif high_act >= precio_tp:
            p_salida, motivo = precio_tp, "TP"
            salida = True
        elif conteo_velas >= MAX_VELAS:
            p_salida, motivo = close_act, "Time Exit"
            salida = True

        if salida:
            pnl = (p_salida - p_entrada) * size
            trades.append({
                "Fecha":      fecha_in,
                "Precio_In":  p_entrada,
                "Precio_Out": p_salida,
                "PnL":        pnl,
                "Motivo":     motivo,
                "MAE":        p_entrada - mae_temp,
                "MFE":        mfe_temp  - p_entrada,
                "Velas":      conteo_velas,
                "Size":       size,
                "SL":         precio_sl,
                "TP":         precio_tp,
            })
            en_posicion = False

rendimiento = pd.DataFrame(trades)
rendimiento["Equity"] = rendimiento["PnL"].cumsum() + CAPITAL
print(f"\nTotal operaciones registradas: {len(rendimiento)}")
rendimiento.head()


## 5. Evaluación Estadística de la Estrategia


In [ ]:
def calcular_metricas(rendimiento, capital_inicial):
    """Calcula las métricas estándar de evaluación de estrategias."""
    if len(rendimiento) == 0:
        return {}

    wins  = rendimiento[rendimiento["PnL"] > 0]["PnL"]
    loses = rendimiento[rendimiento["PnL"] <= 0]["PnL"]

    win_rate      = len(wins) / len(rendimiento)
    avg_win       = wins.mean()  if len(wins)  > 0 else 0
    avg_loss      = loses.mean() if len(loses) > 0 else 0
    profit_factor = wins.sum() / abs(loses.sum()) if loses.sum() != 0 else np.inf
    expectancy    = win_rate * avg_win + (1 - win_rate) * avg_loss

    equity        = rendimiento["Equity"]
    peak          = equity.cummax()
    drawdown      = (equity - peak) / peak * 100
    max_drawdown  = drawdown.min()

    retornos      = rendimiento["PnL"] / capital_inicial
    sharpe        = retornos.mean() / retornos.std() * np.sqrt(252) if retornos.std() > 0 else 0

    pnl_total     = rendimiento["PnL"].sum()
    retorno_total = pnl_total / capital_inicial * 100

    return {
        "Total operaciones": len(rendimiento),
        "Win Rate":          f"{win_rate:.1%}",
        "Profit Factor":     f"{profit_factor:.2f}",
        "Expectancy (USD)":  f"${expectancy:.2f}",
        "Avg Win (USD)":     f"${avg_win:.2f}",
        "Avg Loss (USD)":    f"${avg_loss:.2f}",
        "Max Drawdown":      f"{max_drawdown:.2f}%",
        "Sharpe Ratio":      f"{sharpe:.3f}",
        "PnL Total (USD)":   f"${pnl_total:.2f}",
        "Retorno Total":     f"{retorno_total:.2f}%",
        "Capital Final":     f"${rendimiento['Equity'].iloc[-1]:,.2f}",
    }

metricas = calcular_metricas(rendimiento, CAPITAL)
print("\n" + "="*45)
print("     REPORTE DE RENDIMIENTO DE LA ESTRATEGIA")
print("="*45)
for k, v in metricas.items():
    print(f"  {k:<25} {v}")
print("="*45)


## 6. Visualización de Resultados


In [ ]:
fig = plt.figure(figsize=(16, 12))
fig.suptitle(f"Análisis de Estrategia — {TICKET} | Capital inicial: ${CAPITAL:,}",
             fontsize=14, fontweight="bold", y=0.98)
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

x = rendimiento.index

# ── 1. Curva de Equity ───────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.plot(x, rendimiento["Equity"], color="royalblue", linewidth=1.5, label="Equity")
ax1.axhline(y=CAPITAL, color="gray", linestyle="--", linewidth=1, alpha=0.7, label=f"Capital inicial ${CAPITAL:,}")
ax1.fill_between(x, rendimiento["Equity"], CAPITAL,
                 where=(rendimiento["Equity"] >= CAPITAL), alpha=0.15, color="green")
ax1.fill_between(x, rendimiento["Equity"], CAPITAL,
                 where=(rendimiento["Equity"] < CAPITAL), alpha=0.15, color="red")
ax1.set_title("Curva de Equity", fontweight="bold")
ax1.set_xlabel("Número de operaciones")
ax1.set_ylabel("Capital (USD)")
ax1.legend()
ax1.grid(True, linestyle=":", alpha=0.6)

# ── 2. Distribución de PnL ──────────────────────────────────────
ax2 = fig.add_subplot(gs[1, 0])
colores = ["green" if p > 0 else "red" for p in rendimiento["PnL"]]
ax2.bar(x, rendimiento["PnL"], color=colores, alpha=0.7, width=0.8)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_title("PnL por Operación", fontweight="bold")
ax2.set_xlabel("Número de operaciones")
ax2.set_ylabel("PnL (USD)")
ax2.grid(True, linestyle=":", alpha=0.6)

# ── 3. Distribución de motivos de salida ────────────────────────
ax3 = fig.add_subplot(gs[1, 1])
conteo_motivos = rendimiento["Motivo"].value_counts()
colores_motivo = {"TP": "green", "SL": "red", "Time Exit": "orange"}
bars = ax3.bar(conteo_motivos.index,
               conteo_motivos.values,
               color=[colores_motivo.get(m, "gray") for m in conteo_motivos.index],
               alpha=0.8)
for bar, val in zip(bars, conteo_motivos.values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha="center", fontsize=11, fontweight="bold")
ax3.set_title("Motivos de Salida", fontweight="bold")
ax3.set_ylabel("Número de operaciones")
ax3.grid(True, linestyle=":", alpha=0.6, axis="y")

plt.savefig("resultados_estrategia.png", dpi=150, bbox_inches="tight")
plt.show()
print("Gráfico guardado como 'resultados_estrategia.png'")


## 7. Exportar Resultados


In [ ]:
# Exportar trades detallados
rendimiento.to_csv("rendimiento_detallado.csv", index=True)

# Exportar resumen de métricas
pd.Series(metricas).to_csv("metricas_estrategia.csv", header=["Valor"])

print("Archivos exportados:")
print(" - rendimiento_detallado.csv  (trades completos)")
print(" - metricas_estrategia.csv    (resumen de KPIs)")
print(" - resultados_estrategia.png  (visualización)")


## 8. Conclusiones

### Interpretación de resultados

Los resultados del backtest permiten evaluar la viabilidad estadística de la estrategia:

- **Profit Factor > 1.5** indica que las ganancias superan las pérdidas en una proporción favorable.
- **Win Rate** menor al 50% es aceptable si el ratio R:R es suficientemente alto (aquí TP/SL = 1:1.68).
- **Max Drawdown** mide el peor escenario de pérdida sostenida — un valor < 20% es generalmente aceptable para estrategias de riesgo controlado.
- **Sharpe Ratio > 1.0** indicaría un retorno ajustado al riesgo competitivo.

### Limitaciones del modelo

1. **Slippage y comisiones** no están modelados — en la práctica reducen el rendimiento real.
2. El backtest es **In-Sample** sobre datos históricos completos; se recomienda validación Out-of-Sample.
3. La estrategia es **long-only** — no captura movimientos bajistas.
4. El universo de activos está limitado a un solo ticker; se requiere validación en múltiples activos.

### Próximos pasos

- [ ] Validación Out-of-Sample (split 70/30)
- [ ] Walk-Forward Analysis para detectar sobreajuste
- [ ] Optimización de parámetros con grid search
- [ ] Extensión a múltiples tickers Small Cap
- [ ] Incorporar costos de transacción realistas
